# Airline customer satisfaction: reproducible analysis

This is a **new reproduction notebook**, not the original notebook mentioned in the project context (that file was not supplied). It runs the same versioned workflow as `src/airline_satisfaction/analysis.py`, using only the supplied training CSV. Its holdout is a stratified split of that CSV, not the missing official test set. Preprocessing and PCA are fit within training-only pipelines. See [`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md) for the implementation choices and limitations.

## Run the analysis

The workflow writes aggregate CSV/JSON tables to `reports/tables/` and figures to `reports/figures/`. It does not export passenger-level records.

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from airline_satisfaction.analysis import run

results = run(ROOT / "data" / "raw" / "train.csv", ROOT / "reports")
print(f"Dataset: {results['profile']['rows']:,} rows; {results['profile']['columns_in_source']} source columns")
print(f"Missing values: {results['profile']['missing_values']}")
print(f"Official test file used: {results['profile']['split']['official_test_file_used']}")

Dataset: 103,904 rows; 25 source columns
Missing values: {'Arrival Delay in Minutes': 310}
Official test file used: False


## Holdout model metrics

The labels are encoded as `satisfied = 1` and `neutral or dissatisfied = 0`. All scores below are measured on the stratified 20% holdout created from the supplied training CSV. The third, sensitivity-only model includes the paper-table `TOTAL`, derived as the sum of the 14 service-rating items; the primary baseline excludes this redundant aggregate. KMO/Bartlett suitability diagnostics and PCA are reported. A tested 10-factor scikit-learn fit did not converge under the documented training-only configuration, so no common-factor prediction score is presented.

In [2]:
for model in results["modeling"]["models"]:
    print(model["name"])
    for metric in ("accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc"):
        print(f"  {metric}: {model[metric]:.4f}")
    print("  per class:")
    for label in ("neutral or dissatisfied", "satisfied"):
        values = model["class_report"][label]
        print(f"    {label}: precision={values['precision']:.4f}, recall={values['recall']:.4f}, f1={values['f1-score']:.4f}")
    print(f"  confusion matrix: {model['confusion_matrix']}")
    print(f"  three-fold CV accuracy: {model['cross_validation']['metrics']['accuracy']['mean']:.4f} ± {model['cross_validation']['metrics']['accuracy']['std']:.4f}")

original_features_logistic_regression
  accuracy: 0.8766
  precision_macro: 0.8756
  recall_macro: 0.8725
  f1_macro: 0.8739
  roc_auc: 0.9281
  per class:
    neutral or dissatisfied: precision=0.8819, recall=0.9032, f1=0.8924
    satisfied: precision=0.8693, recall=0.8419, f1=0.8554
  confusion matrix: [[10636, 1140], [1424, 7581]]
  three-fold CV accuracy: 0.8753 ± 0.0009
pca_10_logistic_regression
  accuracy: 0.8455
  precision_macro: 0.8441
  recall_macro: 0.8402
  f1_macro: 0.8418
  roc_auc: 0.9065
  per class:
    neutral or dissatisfied: precision=0.8522, recall=0.8800, f1=0.8659
    satisfied: precision=0.8361, recall=0.8003, f1=0.8178
  confusion matrix: [[10363, 1413], [1798, 7207]]
  three-fold CV accuracy: 0.8445 ± 0.0014
paper_table_feature_sensitivity_with_TOTAL
  accuracy: 0.8771
  precision_macro: 0.8761
  recall_macro: 0.8730
  f1_macro: 0.8744
  roc_auc: 0.9281
  per class:
    neutral or dissatisfied: precision=0.8822, recall=0.9038, f1=0.8929
    satisfied: precisi

## Factor/PCA suitability diagnostics

KMO and Bartlett diagnostics use service-rating items from the training partition only. PCA is a variance-reduction method, not common-factor analysis; neither diagnostic nor the PCA model is claimed to reproduce the paper's exact method.

In [3]:
diagnostics = results["factor_analysis"]
print(f"KMO: {diagnostics['overall_kmo']:.4f}")
print(f"Bartlett p-value: {diagnostics['bartlett_p_value']:.6g}")
print(f"PCA variance explained by first 10 rating components: {diagnostics['pca_cumulative_variance_first_10']:.4f}")
print(diagnostics["caveat"])

KMO: 0.7811
Bartlett p-value: 0
PCA variance explained by first 10 rating components: 0.9242
KMO and Bartlett diagnostics are calculated on the training partition and service-rating items only. PCA is not common-factor analysis and is not asserted to reproduce the paper's procedure.


## Generated outputs

Plots and tabular exports are saved under `reports/figures/` and `reports/tables/`. The paper's reported results and the new holdout results are kept separate in the exported JSON and in [`docs/FINDINGS.md`](../docs/FINDINGS.md). Coefficients and group differences are associations, not causal effects.

In [4]:
print("Figures:")
for path in sorted((ROOT / "reports" / "figures").glob("*.png")):
    print(f"- {path.relative_to(ROOT)}")
print("\nTables:")
for path in sorted((ROOT / "reports" / "tables").glob("*")):
    print(f"- {path.relative_to(ROOT)}")

Figures:
- reports\figures\age_distribution.png
- reports\figures\class_balance.png
- reports\figures\confusion_matrices.png
- reports\figures\delay_distributions_and_satisfaction.png
- reports\figures\feature_correlation_matrix.png
- reports\figures\flight_distance_distribution.png
- reports\figures\missing_values.png
- reports\figures\roc_curves.png
- reports\figures\satisfaction_by_groups.png
- reports\figures\service_rating_correlations.png

Tables:
- reports\tables\class_balance.csv
- reports\tables\data_profile.json
- reports\tables\factor_diagnostics.json
- reports\tables\feature_correlations.csv
- reports\tables\group_comparisons.csv
- reports\tables\group_counts.csv
- reports\tables\missing_values.csv
- reports\tables\model_metrics.json
- reports\tables\satisfaction_by_class.csv
- reports\tables\satisfaction_by_customer_type.csv
- reports\tables\satisfaction_by_gender.csv
- reports\tables\satisfaction_by_type_of_travel.csv
- reports\tables\service_rating_correlations.csv
- rep